In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# display settings
pd.options.display.float_format = '{:.2f}'.format


## Structured Dataset Exploration

In [7]:
df = pd.read_csv(f'D:\\retail-projects\\online-retail-analysis\\data\\raw\\online_retail_II.csv')
df.head()   

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom


In [ ]:
# check the shape of the dataset
df.shape

(1067371, 8)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [ ]:
# check types
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

In [ ]:
# stats summary
df.describe()

,Quantity,Price,Customer ID
count,1067371.00,1067371.00,824364.00
mean,9.94,4.65,15324.64
std,172.71,123.55,1697.46
min,-80995.00,-53594.36,12346.00
25%,1.00,1.25,13975.00
50%,3.00,2.10,15255.00
75%,10.00,4.15,16797.00
max,80995.00,38970.00,18287.00


In [ ]:
# check for missing values
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [15]:
# check for duplicates 
df.duplicated().sum()

np.int64(34335)

##### Granularity: Order-Level vs. Line-Item Level

---

#### 1. Defining the Levels
* **Order-Level Granularity:** Each row represents a complete order. In this case, each `OrderID` is unique across the entire dataset.
* **Line-Item Level Granularity:** Each row represents a specific product within an order. Multiple rows may share the same `OrderID`, but the combination of `OrderID` and `ProductSKU` should be unique.



#### 2. What to Check
To determine the structure of your data, you must verify the uniqueness of your identifiers:

* **For Order-Level Data:** > The number of unique `OrderID` values should equal the total number of rows.
* **For Line-Item Level Data:** > Each `OrderID` appears multiple times, and you should check the count of unique combinations of `OrderID` and `ProductSKU`.

---

#### 3. Action: Checking Granularity in Python
Let’s walk through the code logic to verify which structure your dataset follows.

In [19]:
# Check number of unique invoices
num_invoices = df['Invoice'].nunique()
num_rows = len(df)
print(f"Number of unique invoices: {num_invoices}")
print(f"Total number of rows in dataset: {num_rows}")

Number of unique invoices: 53628
Total number of rows in dataset: 1033036


In [20]:
# Check how many invoices have multiple rows
invoice_counts = df.groupby('Invoice').size()
multi_row_invoices = invoice_counts[invoice_counts > 1]
print(f"Number of invoices with multiple rows (line-item level): {len(multi_row_invoices)}")

Number of invoices with multiple rows (line-item level): 40049


###  Data Structure & Key Observations

* **Rows:** 1,067,371 entries
* **Columns:** 8 columns in total

| Column | Description |
| :--- | :--- |
| **Invoice** | Invoice number. Nominal. A 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter 'c', it indicates a cancellation. |
| **StockCode** | A unique identifier for the product (SKU). |
| **Description** | A short description of the product (missing for some records). |
| **Quantity** | The number of items sold per transaction. |
| **InvoiceDate** | The date and time when the transaction occurred. |
| **Price** | The price of a single unit of the product. |
| **Customer ID** | Unique identifier for each customer (missing for some records). |
| **Country** | The country where the transaction took place. |

---

###  Key Observations

####  Missing Data
* The **Description** column has missing values (around 43,000 records).
* The **Customer ID** column has missing values (around 243,000 records). This is important to note because we might not be able to analyze customer behavior fully without this information.

####  Data Types
* The **Invoice**, **StockCode**, **Description**, and **Country** columns are of type `object` (string-like data).
* The **Quantity** and **Price** columns are numeric (`int64` for Quantity and `float64` for Price).
* The **InvoiceDate** is an `object` type but should be converted to `datetime` for time-based analysis.

####  Statistical Overview
* **Quantity:**
    * **Mean:** ~9.94 items per transaction.
    * **Min:** There are some negative values, which likely indicate returns (this needs further investigation).
    * **Max:** Large values (e.g., ~80,000 items), which could be anomalies or bulk orders.
* **Price:**
    * **Mean:** ~4.65 per item.
    * **Min:** There are some negative price values, which should be further investigated (could indicate data errors or returns).
    * **Max:** Prices go up to around 39,970 per item, which could be outliers or high-ticket items.
* **Customer ID:**
    * There are missing values, which could mean some transactions are not linked to a customer (e.g., guest checkouts or system errors).

#### Duplicates
* There are 34335 row are duplicated in the dataset need to be handeld
  
#### Granularity
* Since many invoices have multiple rows, the dataset is line-item level.

## Data Cleaning

-  Handle missing values in Description and Customer ID (impute or mark as missing).

In [ ]:
print(df["Description"].isnull().sum())    
# Depending on the analysis, we might want to drop rows with missing Description or impute them. For now, let's drop them.
df["Description"].dropna(inplace=True)

4275


In [ ]:
print(df["Customer ID"].isnull().sum())    
# don't drop rows with missing Customer ID for now, as they might still be useful for certain analyses (e.g., overall sales trends).
# We can handle them later if needed.

235151


In [26]:
# Remove duplicates to ensure clean analysis.
df.drop_duplicates(inplace=True)

In [27]:
# Outlier Detection
# Check for outliers in Quantity and Price. might want to investigate or filter out extreme values (e.g., 80,000 items or 39,970 price per unit).
print(df["Quantity"].describe())

count   1033036.00
mean         10.08
std         175.20
min      -80995.00
25%           1.00
50%           3.00
75%          10.00
max       80995.00
Name: Quantity, dtype: float64
